In [0]:
# ============================================================
# CAPA SILVER: Limpieza y deduplicación de datos de telemetría
# ============================================================

# 1. Leer la tabla Bronze que creamos ayer
print("📥 Leyendo tabla bronze_telemetry...")
df_bronze = spark.read.table("bronze_telemetry")
rows_before = df_bronze.count()
print(f"✅ Filas en Bronze: {rows_before}")

# 2. Aplicar reglas de calidad (filtros)
print("\n🧹 Aplicando reglas de calidad...")
df_silver = df_bronze.filter(
    # event_id y device_id no pueden ser nulos
    (df_bronze.event_id.isNotNull()) &
    (df_bronze.device_id.isNotNull()) &
    # Velocidad entre 0 y 200 km/h
    (df_bronze.speed_kmh >= 0) &
    (df_bronze.speed_kmh <= 200) &
    # Temperatura de motor entre -50 y 150 °C
    (df_bronze.engine_temp_c >= -50) &
    (df_bronze.engine_temp_c <= 150) &
    # Batería entre 0 y 100 %
    (df_bronze.battery_pct >= 0) &
    (df_bronze.battery_pct <= 100)
)

rows_after_filters = df_silver.count()
print(f"✅ Filas después de filtros: {rows_after_filters}")
print(f"🗑️  Filas descartadas por calidad: {rows_before - rows_after_filters}")

# 3. Eliminar duplicados (por event_id, que es la clave primaria)
print("\n🔁 Eliminando duplicados por event_id...")
df_silver = df_silver.dropDuplicates(["event_id"])

rows_final = df_silver.count()
print(f"✅ Filas finales en Silver: {rows_final}")
print(f"🗑️  Duplicados eliminados: {rows_after_filters - rows_final}")

# 4. Ver las primeras filas para confirmar que todo se ve bien
print("\n📊 Vista previa de la capa Silver:")
display(df_silver.limit(5))

# 5. Guardar como tabla Delta gestionada "silver_telemetry"
print("\n💾 Guardando como tabla silver_telemetry...")
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_telemetry")

print(f"\n✅ ¡ÉXITO! Tabla silver_telemetry creada con {rows_final} filas limpias.")
print(f"📈 Resumen de calidad: {rows_before} → {rows_final} filas ({round(rows_final/rows_before*100, 2)}% retenidas)")